In [ ]:
# ==============================================================================
# CELL 1: ARCHITECTURE DEPENDENCIES & COMPREHENSIVE ASSET MATRIX
# ==============================================================================
import asyncio
import time
import random
import json
from decimal import Decimal
from typing import Dict, List, Any, Tuple

# Comprehensive Multi-Asset Registry for Chain 137 (61 Assets)
ASSET_MATRIX = [
    "POL", "WPOL", "USDC", "USDC.e", "USDT", "DAI", "WBTC", "WETH", "CRV", "UNI",
    "AAVE", "LINK", "FRAX", "crvUSD", "EUR-0112", "EURS", "jEUR", "PAR", "EURT", "miMATIC",
    "AMUSDT", "AMPOLDAI", "AMPOLUSDC", "RETH", "CBETH", "FRXETH", "SFRXETH", "TBTC", "SOLVBTC", "COMP",
    "SUSHI", "BAL", "QUICK", "KNC", "UMA", "SAND", "MANA", "BAT", "GRT", "SNX",
    "YFI", "COW", "LDO", "ZRO", "TEL", "GEOD", "FLUID", "BUIDL", "EUTBL", "USTBL",
    "OUSG", "BONK", "APE", "PNT", "BUSD", "AUSD", "stBRZ", "BRZ", "BRLA", "FXSwap", "TESOURO"
]

print(f"✅ Asset Matrix verified. Initialized {len(ASSET_MATRIX)} core tokens for Chain 137 evaluation.")

In [ ]:
# ==============================================================================
# CELL 2: CROSS-PROTOCOL MATH ENGINE (V2, V3, CURVE, BALANCER)
# ==============================================================================

class DeFiEngineMath:
    @staticmethod
    def query_uniswap_v2(reserves_a: Decimal, reserves_b: Decimal, amount_in: Decimal, fee: Decimal = Decimal("0.003")) -> Decimal:
        """Standard Constant Product Formula: x * y = k with custom fee parameters."""
        if amount_in <= 0 or reserves_a <= 0 or reserves_b <= 0:
            return Decimal("0")
        amount_in_with_fee = amount_in * (Decimal("1") - fee)
        return (amount_in_with_fee * reserves_b) / (reserves_a + amount_in_with_fee)

    @staticmethod
    def query_uniswap_v3(sqrt_price_x96: Decimal, liquidity: Decimal, amount_in: Decimal, zero_for_one: bool, fee_bps: int) -> Decimal:
        """
        Approximates concentrated liquidity math updates over single tick environments.
        Uses exact custom binary shifts ($2^{96}$) to calculate precise token outcomes.
        """
        if amount_in <= 0 or liquidity <= 0:
            return Decimal("0")

        fee_factor = Decimal("1") - (Decimal(fee_bps) / Decimal("10000"))
        effective_in = amount_in * fee_factor

        # Real-world Tick Math Representation via Bitwise Scaled Multipliers
        price_scale = sqrt_price_x96 / Decimal(2**96)
        current_price = price_scale * price_scale

        if zero_for_one:
            # Token 0 -> Token 1 (Multiplied by Spot Delta)
            return effective_in * current_price
        else:
            # Token 1 -> Token 0 (Divided by Spot Delta)
            return effective_in / current_price if current_price > 0 else Decimal("0")

    @staticmethod
    def query_curve_stable(reserves: List[Decimal], amounts_in: List[Decimal], i: int, j: int, A: Decimal = Decimal("100")) -> Decimal:
        """
        Approximates the Curve StableSwap Invariant over N-dimensions.
        Blends Constant Product and Constant Sum spaces based on amplification factor A.
        """
        if sum(amounts_in) <= 0:
            return Decimal("0")

        # Base implementation tracking balance amplification vectors
        fee = Decimal("0.0004")
        inp = amounts_in[i] * (Decimal("1") - fee)
        out = inp * (reserves[j] / reserves[i]) * (Decimal("1") + (Decimal("1") / A))
        return min(out, reserves[j] * Decimal("0.9"))  # Limit max liquidity drawdown to prevent pool depletion

    @staticmethod
    def query_balancer_weighted(reserves: List[Decimal], weights: List[Decimal], amount_in: Decimal, i: int, j: int, swap_fee: Decimal = Decimal("0.0025")) -> Decimal:
        """
        Implements Balancer Out-In Weighted Invariant formula:
        Out = BalanceOut * (1 - (BalanceIn / (BalanceIn + AmountIn * (1 - Fee))) ^ (WeightIn / WeightOut))
        """
        if amount_in <= 0 or reserves[i] <= 0:
            return Decimal("0")

        effective_in = amount_in * (Decimal("1") - swap_fee)
        weight_ratio = weights[i] / weights[j]
        base = reserves[i] / (reserves[i] + effective_in)

        # Avoid mathematical float underflows inside high-dimension pools
        exponent = Decimal(str(pow(float(base), float(weight_ratio))))
        return reserves[j] * (Decimal("1") - exponent)

print("⚡ Mathematical invariant libraries compiled for V2, V3, Curve, and Balancer structures.")

In [ ]:
# ==============================================================================
# CELL 3: REAL-TIME SIMULATION & ORCHESTRATED RADAR RUNTIME
# ==============================================================================

class DynamicAQSMatrixScanner:
    def __init__(self, assets: List[str]):
        self.assets = assets
        self.macro_interval = 15.0
        self.memory_cache: Dict[str, Dict[str, Any]] = {}
        self.pools: Dict[str, Dict[str, Any]] = {}
        self.metrics = {"ticks_evaluated": 0, "signals_caught": 0}
        self.initialize_mock_pools()

    def initialize_mock_pools(self):
        """Constructs an interoperable multi-protocol ecosystem across our asset registry."""
        random.seed(42)  # Strict seeded state transitions for verification

        # 1. Setup V2 Pools (QuickSwap / Uni V2 Topology)
        for idx in range(0, 20, 2):
            t1, t2 = self.assets[idx], self.assets[idx + 1]
            pool_id = f"V2_{t1}_{t2}"
            self.pools[pool_id] = {
                "protocol": "UniswapV2", "tokens": [t1, t2],
                "reserves": [Decimal(random.randint(50000, 500000)), Decimal(random.randint(50000, 500000))],
                "fee": Decimal("0.003")
            }

        # 2. Setup V3 Pools (Concentrated Liquidity Vectors)
        v3_tiers = [100, 500, 3000]
        for idx in range(10, 35, 2):
            t1, t2 = self.assets[idx], self.assets[idx + 1]
            tier = random.choice(v3_tiers)
            pool_id = f"V3_{t1}_{t2}_{tier}"
            self.pools[pool_id] = {
                "protocol": "UniswapV3", "tokens": [t1, t2],
                "sqrtPriceX96": Decimal(random.randint(70000000000000000000000000000, 85000000000000000000000000000)),
                "liquidity": Decimal(random.randint(1000000000000000000, 50000000000000000000)),
                "fee_bps": tier
            }

        # 3. Setup Curve Multi-Asset Stable/Crypto Pools
        self.pools["CURVE_3POOL_USD"] = {
            "protocol": "Curve", "tokens": ["USDC", "USDT", "DAI"],
            "reserves": [Decimal("10000000"), Decimal("10000000"), Decimal("10000000")], "A": Decimal("200")
        }
        self.pools["CURVE_4EUR_POOL"] = {
            "protocol": "Curve", "tokens": ["jEUR", "PAR", "EURS", "EURT"],
            "reserves": [Decimal("500000"), Decimal("500000"), Decimal("500000"), Decimal("500000")], "A": Decimal("150")
        }

        # 4. Setup Balancer Weighted Multi-Asset Ecosystem Index
        self.pools["BALANCER_8_ASSET_INDEX"] = {
            "protocol": "Balancer",
            "tokens": ["WPOL", "WBTC", "WETH", "LINK", "AAVE", "UNI", "CRV", "BAL"],
            "reserves": [Decimal("200000"), Decimal("50"), Decimal("1000"), Decimal("5000"), Decimal("1000"), Decimal("3000"), Decimal("10000"), Decimal("5000")],
            "weights": [Decimal("0.25"), Decimal("0.15"), Decimal("0.15"), Decimal("0.10"), Decimal("0.05"), Decimal("0.10"), Decimal("0.10"), Decimal("0.10")],
            "swap_fee": Decimal("0.0025")
        }

    async def micro_event_listener(self):
        """Simulates rapid sub-second block changes and transactions inside the 15s window."""
        print("📡 [MICRO-LISTENER] Mounted active real-time memory pipeline...")
        while True:
            # Pick a random pool to experience a price shift inside the current block space
            pool_id = random.choice(list(self.pools.keys()))
            pool = self.pools[pool_id]
            current_time = time.time()

            if pool["protocol"] == "UniswapV2":
                pool["reserves"][0] *= Decimal(str(random.uniform(0.985, 1.015)))
                self.memory_cache[pool_id] = {"type": "RESERVES_UPDATE", "pool": pool, "timestamp": current_time}

            elif pool["protocol"] == "UniswapV3":
                pool["sqrtPriceX96"] *= Decimal(str(random.uniform(0.991, 1.009)))
                self.memory_cache[pool_id] = {"type": "TICK_STMT_UPDATE", "pool": pool, "timestamp": current_time}

            elif pool["protocol"] == "Curve":
                idx = random.randint(0, len(pool["tokens"]) - 1)
                pool["reserves"][idx] *= Decimal(str(random.uniform(0.995, 1.005)))
                self.memory_cache[pool_id] = {"type": "METAPOOL_REBALANCE", "pool": pool, "timestamp": current_time}

            elif pool["protocol"] == "Balancer":
                idx = random.randint(0, len(pool["tokens"]) - 1)
                pool["reserves"][idx] *= Decimal(str(random.uniform(0.99, 1.01)))
                self.memory_cache[pool_id] = {"type": "WEIGHTED_STATE_UPDATE", "pool": pool, "timestamp": current_time}

            await asyncio.sleep(0.2)  # High frequency tick ingestion interval

    async def macro_black_scan_loop(self):
        """Executes deep valuation audit vectors precisely on 15-second intervals."""
        print(f"🦅 [MACRO-SCANNER] Deep Matrix Black Scan Loop Online. Interval: {self.macro_interval}s.")
        await asyncio.sleep(1.0)  # Grace period for network caching initialization

        while self.metrics["ticks_evaluated"] < 3:  # Let it trace 3 consecutive macro iterations for presentation
            start_time = time.time()
            self.metrics["ticks_evaluated"] += 1

            print(f"\n⏱️ [BLACK SCAN TICK #{self.metrics['ticks_evaluated']}] Evaluating Core Trailing Window Across Active Cache Framework...")

            cached_events = list(self.memory_cache.items())
            if not cached_events:
                print("  ↳ [INFO] No state entries discovered inside this interval cycle.")
            else:
                print(f"  ↳ [PROCESSING] Scrutinizing {len(cached_events)} cross-protocol pipeline mutations...")
                for pool_id, update in cached_events:
                    pool = update["pool"]
                    age = start_time - update["timestamp"]

                    if age <= self.macro_interval:
                        # Process multi-protocol variants based on specific underlying math routes
                        if pool["protocol"] == "UniswapV2":
                            out = DeFiEngineMath.query_uniswap_v2(pool["reserves"][0], pool["reserves"][1], Decimal("1000"), pool["fee"])
                            print(f"    🌟 [V2 MATCH] {pool_id} | Out-Quote: {out:.4f} | Cache Age: {age:.2f}s")

                        elif pool["protocol"] == "UniswapV3":
                            out = DeFiEngineMath.query_uniswap_v3(pool["sqrtPriceX96"], pool["liquidity"], Decimal("1000"), True, pool["fee_bps"])
                            print(f"    🌟 [V3 MATCH] {pool_id} | Out-Quote: {out:.4f} | Cache Age: {age:.2f}s")

                        elif pool["protocol"] == "Curve":
                            amounts_in = [Decimal("0")] * len(pool["reserves"])
                            amounts_in[0] = Decimal("1000")  # Swap 1000 base tokens
                            out = DeFiEngineMath.query_curve_stable(pool["reserves"], amounts_in, 0, 1, pool["A"])
                            print(f"    🌟 [CURVE MATCH] {pool_id} | Invariant Out: {out:.4f} | Cache Age: {age:.2f}s")

                        elif pool["protocol"] == "Balancer":
                            out = DeFiEngineMath.query_balancer_weighted(pool["reserves"], pool["weights"], Decimal("1000"), 0, 1, pool["swap_fee"])
                            print(f"    🌟 [BALANCER MATCH] {pool_id} | Weight Out: {out:.4f} | Cache Age: {age:.2f}s")

                        self.metrics["signals_caught"] += 1

                    # Evict entries that have aged out of the trailing execution window
                    if age > self.macro_interval:
                        self.memory_cache.pop(pool_id, None)

            elapsed = time.time() - start_time
            sleep_duration = max(0.1, self.macro_interval - elapsed)
            await asyncio.sleep(sleep_duration)

        print("\n🏁 [EVALUATION COMPLETE] Target macro intervals successfully traced and analyzed inside the notebook engine.")

    async def execute_notebook_demo(self):
        # Tie task threads together into an active concurrent event runner
        listener_task = asyncio.create_task(self.micro_event_listener())
        await self.macro_black_scan_loop()
        listener_task.cancel()  # Safely shutdown loop once execution demo completes


# Initialize and kick-off the entire multi-protocol tracking engine
scanner = DynamicAQSMatrixScanner(ASSET_MATRIX)
await scanner.execute_notebook_demo()

In [ ]:
# ==============================================================================
# CELL 4: CROSS-POOL PRICE RANKER
# Computes effective exchange rates for every token pair across all active pools,
# then ranks pools by best rate per direction to surface the highest-value routing.
# ==============================================================================
import math
from collections import defaultdict

QUOTE_AMOUNT = Decimal("1000")  # Normalised input amount for all rate comparisons


def compute_all_pool_rates(pools: dict) -> dict:
    """
    Iterates every pool and computes the effective exchange rate
    (amount_out / amount_in) for each directional token pair it supports.

    Returns:
        rates[(token_in, token_out)] = [
            {"pool_id": str, "protocol": str, "rate": Decimal, "amount_out": Decimal},
            ...
        ]
    """
    rates: dict = defaultdict(list)

    for pool_id, pool in pools.items():
        proto = pool["protocol"]
        tokens = pool["tokens"]
        n = len(tokens)

        if proto == "UniswapV2":
            # Both directions for the single pair
            r = pool["reserves"]
            fee = pool["fee"]
            for (i, j) in [(0, 1), (1, 0)]:
                out = DeFiEngineMath.query_uniswap_v2(r[i], r[j], QUOTE_AMOUNT, fee)
                if out > 0:
                    rates[(tokens[i], tokens[j])].append({
                        "pool_id": pool_id, "protocol": proto,
                        "rate": out / QUOTE_AMOUNT, "amount_out": out
                    })

        elif proto == "UniswapV3":
            sq = pool["sqrtPriceX96"]
            liq = pool["liquidity"]
            fb = pool["fee_bps"]
            for (i, j, z4o) in [(0, 1, True), (1, 0, False)]:
                out = DeFiEngineMath.query_uniswap_v3(sq, liq, QUOTE_AMOUNT, z4o, fb)
                if out > 0:
                    rates[(tokens[i], tokens[j])].append({
                        "pool_id": pool_id, "protocol": proto,
                        "rate": out / QUOTE_AMOUNT, "amount_out": out
                    })

        elif proto == "Curve":
            r = pool["reserves"]
            A = pool["A"]
            for i in range(n):
                for j in range(n):
                    if i == j:
                        continue
                    amounts_in = [Decimal("0")] * n
                    amounts_in[i] = QUOTE_AMOUNT
                    out = DeFiEngineMath.query_curve_stable(r, amounts_in, i, j, A)
                    if out > 0:
                        rates[(tokens[i], tokens[j])].append({
                            "pool_id": pool_id, "protocol": proto,
                            "rate": out / QUOTE_AMOUNT, "amount_out": out
                        })

        elif proto == "Balancer":
            r = pool["reserves"]
            w = pool["weights"]
            sf = pool["swap_fee"]
            for i in range(n):
                for j in range(n):
                    if i == j:
                        continue
                    out = DeFiEngineMath.query_balancer_weighted(r, w, QUOTE_AMOUNT, i, j, sf)
                    if out > 0:
                        rates[(tokens[i], tokens[j])].append({
                            "pool_id": pool_id, "protocol": proto,
                            "rate": out / QUOTE_AMOUNT, "amount_out": out
                        })

    # Sort each pair's pool list by rate descending (best rate first)
    for pair in rates:
        rates[pair].sort(key=lambda x: x["rate"], reverse=True)

    return dict(rates)


def print_ranked_rates(rates: dict, top_n: int = 5):
    """
    Prints a ranked table for every token pair that has more than one pool quote,
    i.e. pairs where cross-pool comparison is meaningful.
    """
    multi_pool_pairs = {k: v for k, v in rates.items() if len(v) >= 2}
    print(f"📊 CROSS-POOL PRICE RANKING REPORT")
    print(f"   Pairs with multi-pool coverage: {len(multi_pool_pairs)} | "
          f"Total directional quotes: {sum(len(v) for v in rates.values())}")
    print("=" * 90)

    for (token_in, token_out), pool_list in sorted(
        multi_pool_pairs.items(), key=lambda x: x[0]
    ):
        best = pool_list[0]
        worst = pool_list[-1]
        spread_pct = float((best["rate"] - worst["rate"]) / worst["rate"] * 100) if worst["rate"] > 0 else 0.0
        print(f"\n  {token_in:>12s} → {token_out:<12s}  "
              f"[{len(pool_list)} pools | spread: {spread_pct:+.4f}%]")
        print(f"  {'Rank':<5} {'Pool ID':<40} {'Protocol':<12} {'Rate':>12} {'Δ vs Best':>12}")
        print(f"  {'-'*85}")
        for rank, entry in enumerate(pool_list[:top_n], 1):
            delta_pct = float((entry["rate"] - best["rate"]) / best["rate"] * 100)
            marker = "★ BEST" if rank == 1 else ("▼ WORST" if rank == len(pool_list[:top_n]) else "")
            print(f"  #{rank:<4} {entry['pool_id']:<40} {entry['protocol']:<12} "
                  f"{float(entry['rate']):>12.6f} {delta_pct:>+11.4f}%  {marker}")

    print("\n" + "=" * 90)
    print(f"✅ Price ranking complete across {len(rates)} directional token-pair routes.")


# Re-initialise the scanner pool state (seeded deterministically)
scanner = DynamicAQSMatrixScanner(ASSET_MATRIX)
ALL_RATES = compute_all_pool_rates(scanner.pools)
print_ranked_rates(ALL_RATES)


In [ ]:
# ==============================================================================
# CELL 5: ARBITRAGE PATH GRAPH + BELLMAN-FORD CYCLE DETECTOR
# Builds a directed weighted graph where nodes = tokens and
# edge weight = -log(rate).  Negative-weight cycles in this graph
# correspond exactly to profitable arbitrage loops.
# ==============================================================================
import math


class ArbitrageGraphEngine:
    """
    Constructs and analyses a directed exchange-rate graph for arbitrage detection.

    Graph representation
    --------------------
    Nodes  : token symbols
    Edges  : (token_in, token_out, weight=-log(rate), pool_id, protocol)

    Arbitrage condition
    -------------------
    A cycle  t0 → t1 → … → tn → t0  is profitable when
        ∏ rate_i  >  1   ⟺   Σ (-log(rate_i))  <  0
    i.e. a *negative-weight cycle* in the transformed graph.
    Bellman-Ford detects these in O(V·E) time.
    """

    def __init__(self, rates: dict):
        self.edges: List[Tuple] = []   # (u, v, weight, pool_id, protocol, rate)
        self.tokens: List[str] = []
        self._build_graph(rates)

    def _build_graph(self, rates: dict):
        token_set = set()
        for (tin, tout), pool_list in rates.items():
            token_set.add(tin)
            token_set.add(tout)
            # Use best rate per (token_in, token_out, pool) — one edge per pool
            for entry in pool_list:
                r = float(entry["rate"])
                if r <= 0:
                    continue
                weight = -math.log(r)   # Negative log-rate transformation
                self.edges.append((tin, tout, weight, entry["pool_id"], entry["protocol"], r))
        self.tokens = sorted(token_set)
        print(f"📐 Graph built: {len(self.tokens)} nodes (tokens), {len(self.edges)} edges (pool routes)")

    def bellman_ford_all_sources(self) -> List[dict]:
        """
        Runs Bellman-Ford from every token as source.
        Collects all unique negative-weight cycles (arbitrage loops).
        Returns a list of opportunity dicts, deduplicated by canonical cycle signature.
        """
        token_idx = {t: i for i, t in enumerate(self.tokens)}
        n = len(self.tokens)
        opportunities = {}

        for source in self.tokens:
            dist = {t: float("inf") for t in self.tokens}
            pred = {t: None for t in self.tokens}          # predecessor token
            pred_edge = {t: None for t in self.tokens}     # edge metadata
            dist[source] = 0.0

            # Relax edges n-1 times
            for _ in range(n - 1):
                updated = False
                for (u, v, w, pid, proto, rate) in self.edges:
                    if dist[u] != float("inf") and dist[u] + w < dist[v]:
                        dist[v] = dist[u] + w
                        pred[v] = u
                        pred_edge[v] = {"pool_id": pid, "protocol": proto, "rate": rate}
                        updated = True
                if not updated:
                    break

            # n-th relaxation — any further improvement reveals a negative cycle
            for (u, v, w, pid, proto, rate) in self.edges:
                if dist[u] != float("inf") and dist[u] + w < dist[v]:
                    # Trace the cycle back through predecessors
                    cycle_node = v
                    visited = set()
                    # Walk back n steps to guarantee we are inside the cycle
                    for _ in range(n):
                        cycle_node = pred[cycle_node]
                        if cycle_node is None:
                            break

                    if cycle_node is None:
                        continue

                    # Extract the cycle path
                    path = []
                    edge_path = []
                    cur = cycle_node
                    while True:
                        if cur in visited:
                            # Trim to the actual cycle start
                            idx = path.index(cur)
                            path = path[idx:]
                            edge_path = edge_path[idx:]
                            break
                        visited.add(cur)
                        path.append(cur)
                        edge_path.append(pred_edge[cur])
                        nxt = pred[cur]
                        if nxt is None:
                            break
                        cur = nxt

                    if len(path) < 2:
                        continue

                    # Canonical key: smallest-rotation of the cycle (deduplication)
                    cycle_key = tuple(min(
                        [path[i:] + path[:i] for i in range(len(path))],
                        key=lambda x: x
                    ))
                    if cycle_key in opportunities:
                        continue

                    # Compute cumulative product of rates around the cycle
                    cum_rate = 1.0
                    for ep in edge_path:
                        if ep:
                            cum_rate *= ep["rate"]

                    profit_pct = (cum_rate - 1.0) * 100.0
                    opportunities[cycle_key] = {
                        "path": path + [path[0]],
                        "edges": edge_path,
                        "cumulative_rate": cum_rate,
                        "profit_pct": profit_pct,
                    }

        return sorted(opportunities.values(), key=lambda x: x["profit_pct"], reverse=True)


def print_arb_report(opportunities: List[dict], max_show: int = 20):
    if not opportunities:
        print("  ↳ No profitable arbitrage cycles detected in the current pool state.")
        return

    print(f"\n{'='*90}")
    print(f"🔥 ARBITRAGE OPPORTUNITY REPORT  —  {len(opportunities)} unique cycle(s) detected")
    print(f"{'='*90}")

    for rank, opp in enumerate(opportunities[:max_show], 1):
        path_str = " → ".join(opp["path"])
        hops = len(opp["path"]) - 1
        profit = opp["profit_pct"]
        cum = opp["cumulative_rate"]
        tier = "🟢 STRONG" if profit > 1.0 else ("🟡 MARGINAL" if profit > 0.1 else "🔴 MICRO")
        print(f"\n  #{rank:>3}  {tier}")
        print(f"       Path ({hops} hop{'s' if hops > 1 else ''}): {path_str}")
        print(f"       Cumulative Rate: {cum:.8f}   Gross Profit: {profit:+.6f}%")
        print(f"       Pool Sequence:")
        for ep in opp["edges"]:
            if ep:
                print(f"         ├─ [{ep['protocol']:<12}] {ep['pool_id']}  rate={ep['rate']:.6f}")

    if len(opportunities) > max_show:
        print(f"\n  … and {len(opportunities) - max_show} additional cycles (truncated).")

    print(f"\n{'='*90}")
    best = opportunities[0]
    print(f"  ★  Best gross opportunity: {best['profit_pct']:+.6f}% on path: {' → '.join(best['path'])}")
    print(f"     ⚠️  Note: gross profit does not account for gas costs, slippage, or MEV.")
    print(f"{'='*90}\n")


# ------------------------------------------------------------------
# Build the arbitrage graph from the ranked rates computed in Cell 4
# ------------------------------------------------------------------
print("🕸️  Constructing Arbitrage Path Graph from cross-pool rate matrix...")
arb_engine = ArbitrageGraphEngine(ALL_RATES)

print("\n⚡ Running Bellman-Ford negative-cycle detection across all source nodes...")
opportunities = arb_engine.bellman_ford_all_sources()

print(f"\n✅ Detection complete. {len(opportunities)} arbitrage cycle(s) found.")
print_arb_report(opportunities)


In [ ]:
# ==============================================================================
# CELL 6: LIVE RPC LAYER — POLYGON CONNECTION & DEEP POOL REGISTRY
# Connects to Polygon via POLYGON_RPC_URL env var, defines a deep pool address
# registry for major liquidity venues, and implements a multicall-based batch
# state loader that falls back to individual eth_call on rate-limit errors.
# ==============================================================================
import os
import time
from dataclasses import dataclass, field
from typing import Optional
from web3 import Web3
from web3.exceptions import ContractLogicError

# ── RPC Connection ────────────────────────────────────────────────────────────
# Priority: 1) WSS publicnode (real-time)  2) HTTP publicnode  3) env override  4) other public
_WSS_PRIMARY  = os.environ.get("POLYGON_WSS_URL",  "wss://polygon-bor-rpc.publicnode.com")
_HTTP_PRIMARY = os.environ.get("POLYGON_RPC_URL",  "https://polygon-bor-rpc.publicnode.com")
CHAIN_ID = 137

w3       = None
BLOCK    = 0
RPC_LIVE = False

# 1. WSS first — lowest latency, persistent block subscription
try:
    _ws_cand = Web3(Web3.WebsocketProvider(_WSS_PRIMARY, websocket_timeout=15))
    try:
        from web3.middleware import ExtraDataToPOAMiddleware
        _ws_cand.middleware_onion.inject(ExtraDataToPOAMiddleware, layer=0)
    except (ImportError, AttributeError):
        try:
            from web3.middleware import geth_poa_middleware
            _ws_cand.middleware_onion.inject(geth_poa_middleware, layer=0)
        except (ImportError, AttributeError):
            pass
    BLOCK    = _ws_cand.eth.block_number
    w3       = _ws_cand
    RPC_LIVE = True
    print(f"✅ WSS connected  →  {_WSS_PRIMARY.split('/')[2]}  block #{BLOCK:,}")
except Exception as _wss_exc:
    print(f"  ⚠️  WSS [{_WSS_PRIMARY.split('/')[2]}] unavailable: {_wss_exc}")

# 2. HTTP fallback chain if WSS unavailable
if not RPC_LIVE:
    for _url in [_HTTP_PRIMARY, "https://rpc.ankr.com/polygon", "https://polygon-rpc.com"]:
        if not _url:
            continue
        try:
            _cand = Web3(Web3.HTTPProvider(_url, request_kwargs={"timeout": 15}))
            BLOCK    = _cand.eth.block_number
            w3       = _cand
            RPC_LIVE = True
            _lbl     = _url.split("/")[2] if "//" in _url else _url
            print(f"✅ HTTP connected  →  {_lbl}  block #{BLOCK:,}")
            break
        except Exception as _http_exc:
            _lbl = _url.split("/")[2] if "//" in _url else _url
            print(f"  ⚠️  HTTP [{_lbl}] unavailable: {_http_exc}")

if not RPC_LIVE:
    print("⚠️  All RPC endpoints unreachable. System will run in SIMULATION mode.")

POLYGON_RPC_URL = _HTTP_PRIMARY  # retained for backward compatibility

# ── Deep Pool Address Registry (Polygon mainnet, verified) ───────────────────
# Format: pool_id → {protocol, token0, token1, address, fee_bps}
DEEP_POOL_REGISTRY = {
    # ── Uniswap V3 ────────────────────────────────────────────────────────
    "V3_USDC_WETH_500":    {"protocol": "UniswapV3", "token0": "USDC",   "token1": "WETH",  "address": "0x45dDa9cb7c25131DF268515131f647d726f50608", "fee_bps": 500},
    "V3_USDC_WETH_3000":   {"protocol": "UniswapV3", "token0": "USDC",   "token1": "WETH",  "address": "0x0e44cEb592AcFC5D3F09D996302eB4C499ff8c10", "fee_bps": 3000},
    "V3_WBTC_WETH_500":    {"protocol": "UniswapV3", "token0": "WBTC",   "token1": "WETH",  "address": "0x50eaEDB835021E4A108B7290636d62E9765cc6d7", "fee_bps": 500},
    "V3_WPOL_USDC_500":    {"protocol": "UniswapV3", "token0": "WPOL",   "token1": "USDC",  "address": "0xA374094527e1673A86dE625aa59517c5dE346d32", "fee_bps": 500},
    "V3_WPOL_WETH_500":    {"protocol": "UniswapV3", "token0": "WPOL",   "token1": "WETH",  "address": "0x86f1d8390222A3691C28938eC7404A1661E618e0", "fee_bps": 500},
    "V3_USDC_USDT_100":    {"protocol": "UniswapV3", "token0": "USDC",   "token1": "USDT",  "address": "0xDaC8A8E6DBf8c690ec6815e0fF03491B2770255D", "fee_bps": 100},
    "V3_DAI_USDC_100":     {"protocol": "UniswapV3", "token0": "DAI",    "token1": "USDC",  "address": "0x5f69C2ec01c22843f8273838d570243fd1963014", "fee_bps": 100},
    "V3_LINK_WETH_3000":   {"protocol": "UniswapV3", "token0": "LINK",   "token1": "WETH",  "address": "0x3e31AB7f37c048FC6574189135D108df80F0ea26", "fee_bps": 3000},
    "V3_AAVE_WETH_3000":   {"protocol": "UniswapV3", "token0": "AAVE",   "token1": "WETH",  "address": "0x148cE9b50Be946a96e17b4f7D6b2CF34BC0Cb5d9", "fee_bps": 3000},
    "V3_CRV_WETH_3000":    {"protocol": "UniswapV3", "token0": "CRV",    "token1": "WETH",  "address": "0xb0Cc2d5C3Ad83D22FE9bCa715F614B3699c0e21D", "fee_bps": 3000},
    # ── QuickSwap V2 (Uniswap V2 compatible) ─────────────────────────────
    "QS_WPOL_USDC":        {"protocol": "UniswapV2", "token0": "WPOL",   "token1": "USDC",  "address": "0x6e7a5FAFcec6BB1e78bAE2A1F0B612012BF14827", "fee_bps": 30},
    "QS_WPOL_WETH":        {"protocol": "UniswapV2", "token0": "WPOL",   "token1": "WETH",  "address": "0xadbF1854e5883eB8aa7BAf50705338739e558E5b", "fee_bps": 30},
    "QS_WBTC_WETH":        {"protocol": "UniswapV2", "token0": "WBTC",   "token1": "WETH",  "address": "0xdC9232E2Df177d7a12FdFf6EcBAb114E2231198D", "fee_bps": 30},
    "QS_USDC_USDT":        {"protocol": "UniswapV2", "token0": "USDC",   "token1": "USDT",  "address": "0x2cF7252e74036d1Da831d11089D326296e64a728", "fee_bps": 30},
    "QS_USDC_DAI":         {"protocol": "UniswapV2", "token0": "USDC",   "token1": "DAI",   "address": "0xf04adBF75cDFc5eD26eEA4bbbb991DB002036Bdd", "fee_bps": 30},
    "QS_WETH_USDC":        {"protocol": "UniswapV2", "token0": "WETH",   "token1": "USDC",  "address": "0x853Ee4b2A13f8a742d64C8F088bE7bA2131f670d", "fee_bps": 30},
    "QS_LINK_WETH":        {"protocol": "UniswapV2", "token0": "LINK",   "token1": "WETH",  "address": "0x5cA6CA6c3709E1E6CFe74a50Cf6B2B6BA2Dadd67", "fee_bps": 30},
    "QS_AAVE_WETH":        {"protocol": "UniswapV2", "token0": "AAVE",   "token1": "WETH",  "address": "0x90bc3E68Ba8393a3Bf2D79309365089975341a43", "fee_bps": 30},
    # ── Curve Pools ───────────────────────────────────────────────────────
    "CRV_AAVE3POOL":       {"protocol": "Curve",     "token0": "DAI",    "token1": "USDC",  "address": "0x445FE580eF8d70FF569aB36e898ed8b2B4Eee5BE", "fee_bps": 4, "tokens": ["DAI", "USDC", "USDT"], "A": 200},
    "CRV_ATRICRYPTO":      {"protocol": "Curve",     "token0": "USDT",   "token1": "WBTC",  "address": "0x1d8b86e3D88cDb2d34688e87E72F388Cb541B7C8", "fee_bps": 4, "tokens": ["DAI", "USDC", "USDT", "WBTC", "WETH"], "A": 2},
    # ── Balancer V2 ───────────────────────────────────────────────────────
    "BAL_WPOL_WETH_USDC":  {"protocol": "Balancer",  "token0": "WPOL",   "token1": "WETH",  "address": "0x0297e37f1873D2DAb4487Aa67cD56B58E2F27875",
                             "pool_id": "0x0297e37f1873d2dab4487aa67cd56b58e2f27875000100000000000000000002",
                             "tokens": ["WPOL", "WETH", "USDC"], "weights": [0.25, 0.25, 0.50], "fee_bps": 10},
}

TOKEN_ADDRESSES = {
    "WPOL":   "0x0d500B1d8E8eF31E21C99d1Db9A6444d3ADf1270",
    "USDC":   "0x2791Bca1f2de4661ED88A30C99A7a9449Aa84174",
    "USDC.e": "0x2791Bca1f2de4661ED88A30C99A7a9449Aa84174",
    "USDT":   "0xc2132D05D31c914a87C6611C10748AEb04B58e8F",
    "DAI":    "0x8f3Cf7ad23Cd3CaDbD9735AFf958023239c6A063",
    "WBTC":   "0x1BFD67037B42Cf73acF2047067bd4F2C47D9BfD6",
    "WETH":   "0x7ceB23fD6bC0adD59E62ac25578270cFf1b9f619",
    "LINK":   "0x53E0bca35eC356BD5ddDFebbD1Fc0fD03FaBad39",
    "AAVE":   "0xD6DF932A45C0f255f85145f286eA0b292B21C90B",
    "CRV":    "0x172370d5Cd63279eFa6d502DAB29171933a610AF",
    "UNI":    "0xb33EaAd8d922B1083446DC23f610c2567fB5180f",
    "BAL":    "0x9a71012B13CA4d3D0Cdc72A177DF3ef03b0E76A5",
}

# ── ABI Fragments ─────────────────────────────────────────────────────────────
ABI_V2_PAIR = [
    {"name": "getReserves", "type": "function", "stateMutability": "view",
     "inputs": [], "outputs": [
         {"name": "reserve0", "type": "uint112"},
         {"name": "reserve1", "type": "uint112"},
         {"name": "blockTimestampLast", "type": "uint32"}]},
]
ABI_V3_POOL = [
    {"name": "slot0", "type": "function", "stateMutability": "view",
     "inputs": [], "outputs": [
         {"name": "sqrtPriceX96",       "type": "uint160"},
         {"name": "tick",               "type": "int24"},
         {"name": "observationIndex",   "type": "uint16"},
         {"name": "observationCardinality","type": "uint16"},
         {"name": "observationCardinalityNext","type": "uint16"},
         {"name": "feeProtocol",        "type": "uint8"},
         {"name": "unlocked",           "type": "bool"}]},
    {"name": "liquidity", "type": "function", "stateMutability": "view",
     "inputs": [], "outputs": [{"name": "", "type": "uint128"}]},
]
ABI_ERC20_BALANCE = [
    {"name": "balanceOf", "type": "function", "stateMutability": "view",
     "inputs": [{"name": "account", "type": "address"}],
     "outputs": [{"name": "", "type": "uint256"}]},
    {"name": "decimals", "type": "function", "stateMutability": "view",
     "inputs": [], "outputs": [{"name": "", "type": "uint8"}]},
]

# ── Live State Loader ─────────────────────────────────────────────────────────
def load_live_pool_state(pool_id: str, pool_meta: dict) -> Optional[dict]:
    """
    Fetches live state for a single pool from Polygon RPC.
    Returns a pool-state dict compatible with DeFiEngineMath, or None on failure.
    """
    if not RPC_LIVE:
        return None

    addr = pool_meta["address"]
    proto = pool_meta["protocol"]

    try:
        if proto == "UniswapV2":
            c = w3.eth.contract(address=Web3.to_checksum_address(addr), abi=ABI_V2_PAIR)
            r0, r1, _ = c.functions.getReserves().call()
            return {
                "protocol": "UniswapV2",
                "tokens": [pool_meta["token0"], pool_meta["token1"]],
                "reserves": [Decimal(r0), Decimal(r1)],
                "fee": Decimal(pool_meta["fee_bps"]) / Decimal("10000"),
                "address": addr, "pool_id": pool_id,
            }

        elif proto == "UniswapV3":
            c = w3.eth.contract(address=Web3.to_checksum_address(addr), abi=ABI_V3_POOL)
            slot0 = c.functions.slot0().call()
            liq   = c.functions.liquidity().call()
            return {
                "protocol": "UniswapV3",
                "tokens": [pool_meta["token0"], pool_meta["token1"]],
                "sqrtPriceX96": Decimal(slot0[0]),
                "liquidity":    Decimal(liq),
                "fee_bps":      pool_meta["fee_bps"],
                "address": addr, "pool_id": pool_id,
            }

        elif proto == "Curve":
            toks = pool_meta.get("tokens", [pool_meta["token0"], pool_meta["token1"]])
            reserves = []
            for tok in toks:
                tok_addr = TOKEN_ADDRESSES.get(tok)
                if tok_addr:
                    erc = w3.eth.contract(address=Web3.to_checksum_address(tok_addr), abi=ABI_ERC20_BALANCE)
                    bal = erc.functions.balanceOf(Web3.to_checksum_address(addr)).call()
                    reserves.append(Decimal(bal))
                else:
                    reserves.append(Decimal("1000000"))
            return {
                "protocol": "Curve",
                "tokens": toks,
                "reserves": reserves,
                "A": Decimal(str(pool_meta.get("A", 100))),
                "address": addr, "pool_id": pool_id,
            }

        elif proto == "Balancer":
            toks = pool_meta.get("tokens", [pool_meta["token0"], pool_meta["token1"]])
            reserves = []
            for tok in toks:
                tok_addr = TOKEN_ADDRESSES.get(tok)
                if tok_addr:
                    erc = w3.eth.contract(address=Web3.to_checksum_address(tok_addr), abi=ABI_ERC20_BALANCE)
                    bal = erc.functions.balanceOf(Web3.to_checksum_address(addr)).call()
                    reserves.append(Decimal(bal))
                else:
                    reserves.append(Decimal("1000000"))
            w_raw = pool_meta.get("weights", [1.0 / len(toks)] * len(toks))
            return {
                "protocol": "Balancer",
                "tokens": toks,
                "reserves": reserves,
                "weights": [Decimal(str(w)) for w in w_raw],
                "swap_fee": Decimal(pool_meta["fee_bps"]) / Decimal("10000"),
                "address": addr, "pool_id": pool_id,
            }

    except Exception as exc:
        print(f"    ⚠️  [{pool_id}] RPC call failed: {exc}")
        return None


def load_all_live_pools(registry: dict) -> dict:
    """
    Iterates the full deep pool registry, loads live states, and returns a
    combined pool dict.  Falls back to the scanner's mock pools when RPC is
    unavailable so the system degrades gracefully.
    """
    live_pools = {}
    failed = 0

    if RPC_LIVE:
        print(f"🌐 Loading live state for {len(registry)} registered pools...")
        for pool_id, meta in registry.items():
            state = load_live_pool_state(pool_id, meta)
            if state:
                live_pools[pool_id] = state
            else:
                failed += 1
            time.sleep(0.05)   # 20 req/s — polite rate for public endpoints
        print(f"   ✅ {len(live_pools)} pools loaded live  |  ⚠️  {failed} failed")
    else:
        print("📴 RPC offline — loading deterministic simulation pools as fallback...")
        _scanner = DynamicAQSMatrixScanner(ASSET_MATRIX)
        live_pools = _scanner.pools
        print(f"   ↳ {len(live_pools)} simulation pools loaded")

    return live_pools


LIVE_POOLS = load_all_live_pools(DEEP_POOL_REGISTRY)
print(f"\n📦 LIVE_POOLS registry ready: {len(LIVE_POOLS)} pools across "
      f"{len({p['protocol'] for p in LIVE_POOLS.values()})} protocol families")


In [ ]:
# ==============================================================================
# CELL 6B: MULTI-PROVIDER DATA LAYER
# Upgrades the RPC stack with WSS (PublicNode) + Chainstack HTTP fallback.
# Wires in four live price oracles -- Chainlink (on-chain), CoinGecko, 1inch,
# and DexScreener -- and exposes refresh_token_prices() which populates the
# TOKEN_USD_PRICE dict used by every downstream profitability calculation.
# ==============================================================================
import requests
from typing import Dict

# -- Environment Variables ----------------------------------------------------
CHAINSTACK_URL  = os.environ.get("CHAINSTACK_URL",    "https://polygon-mainnet.chainstackapis.com")
WSS_URL         = os.environ.get("POLYGON_WSS_URL",   "wss://polygon-bor-rpc.publicnode.com")
ONEINCH_API_KEY = os.environ.get("ONEINCH_API_KEY",   "")   # free key: portal.1inch.dev
COINGECKO_KEY   = os.environ.get("COINGECKO_API_KEY", "")   # free demo key: coingecko.com/en/developers

# -- 1. RPC Upgrade: HTTP Priority List ----------------------------------------
# Tries providers in order; promotes w3 to the first successful connection.
_HTTP_PRIORITY = [
    "https://polygon-bor-rpc.publicnode.com",   # publicnode – no key required
    os.environ.get("POLYGON_RPC_URL", ""),
    CHAINSTACK_URL,
    "https://rpc.ankr.com/polygon",
    "https://polygon-rpc.com",
]

for _url in _HTTP_PRIORITY:
    if not _url:
        continue
    try:
        _cand = Web3(Web3.HTTPProvider(_url, request_kwargs={"timeout": 10}))
        _blk  = _cand.eth.block_number
        if not RPC_LIVE:
            w3 = _cand
            RPC_LIVE = True
        print(f"\u2705 HTTP  [{_url.split('/')[2]}]  \u2192  block #{_blk:,}")
        break
    except Exception as _e:
        _lbl = _url.split('/')[2] if '//' in _url else _url
        print(f"  \u26a0\ufe0f  HTTP  [{_lbl}]  skipped: {_e}")

# -- 2. WSS Connection: PublicNode Real-time Block Subscription ----------------
w3_wss = None
try:
    _ws = Web3(Web3.WebsocketProvider(WSS_URL, websocket_timeout=15))
    # Inject PoA middleware for Polygon (handles block.extraData length)
    try:
        from web3.middleware import geth_poa_middleware
        _ws.middleware_onion.inject(geth_poa_middleware, layer=0)
    except (ImportError, AttributeError):
        try:
            from web3.middleware import ExtraDataToPOAMiddleware
            _ws.middleware_onion.inject(ExtraDataToPOAMiddleware, layer=0)
        except ImportError:
            pass
    _wb = _ws.eth.block_number
    w3_wss = _ws
    w3 = w3_wss          # promote WSS to primary -- lower read latency
    RPC_LIVE = True
    print(f"\u2705 WSS   [{WSS_URL.split('/')[2]}]  \u2192  block #{_wb:,}")
except Exception as _e:
    _lbl = WSS_URL.split('/')[2] if '//' in WSS_URL else WSS_URL
    print(f"  \u26a0\ufe0f  WSS  [{_lbl}]  unavailable: {_e}")

# -- 3. Chainlink AggregatorV3 On-Chain Oracles (Polygon mainnet) --------------
_CL_ABI = [
    {"inputs": [], "name": "latestRoundData",
     "outputs": [{"name": "roundId",        "type": "uint80"},
                 {"name": "answer",          "type": "int256"},
                 {"name": "startedAt",       "type": "uint256"},
                 {"name": "updatedAt",       "type": "uint256"},
                 {"name": "answeredInRound", "type": "uint80"}],
     "stateMutability": "view", "type": "function"},
    {"inputs": [], "name": "decimals",
     "outputs": [{"name": "", "type": "uint8"}],
     "stateMutability": "view", "type": "function"},
]

CHAINLINK_FEEDS: Dict[str, str] = {
    "WPOL":  "0xAB594600376Ec9fD91F8e885dADF0CE036862dE0",  # MATIC/USD
    "POL":   "0xAB594600376Ec9fD91F8e885dADF0CE036862dE0",
    "WETH":  "0xF9680D99D6C9589e2a93a78A04A279e509205945",  # ETH/USD
    "WBTC":  "0xc907E116054Ad103354f2D350FD2514433D57F6f",  # BTC/USD
    "LINK":  "0xd9FFdb71EbE7496cC440152d43986Aae0AB76665",  # LINK/USD
    "AAVE":  "0x72484B12719E23115761D5DA1646945632979bB6",  # AAVE/USD
    "USDC":  "0xfE4A8cc5b5B2366C1B58Bea3858e81843581b2F7",  # USDC/USD
    "USDT":  "0x0A6513e40db6EB1b165753AD52E80663aeA50545",  # USDT/USD
    "DAI":   "0x4746DeC9e833A82EC7C2C1356372CcF2cfcD2F3D",  # DAI/USD
    "CRV":   "0x336584C8E6Dc19637A5b36206B1c79923111b405",  # CRV/USD
    "UNI":   "0xdf0Fb4e4F928d2dCB76f438575fDD8682386e13C",  # UNI/USD
    "BAL":   "0xD106B538F2A868c28Ca1Ec7E298C3325c0226b1b",  # BAL/USD
    "FRAX":  "0x00DBeB1e45485d53DF7C2F0dF1Aa0b6Dc30311d3",  # FRAX/USD
    "EURS":  "0x73366Fe0AA0Ded304479862808e02506FE556a98",  # EUR/USD
    "EURT":  "0x73366Fe0AA0Ded304479862808e02506FE556a98",
    "jEUR":  "0x73366Fe0AA0Ded304479862808e02506FE556a98",
    "PAR":   "0x73366Fe0AA0Ded304479862808e02506FE556a98",
}

def _chainlink_price(symbol: str):
    """Reads latest USD price from a Chainlink AggregatorV3 feed on Polygon."""
    feed_addr = CHAINLINK_FEEDS.get(symbol)
    if not feed_addr or not RPC_LIVE or w3 is None:
        return None
    try:
        contract = w3.eth.contract(
            address=Web3.to_checksum_address(feed_addr), abi=_CL_ABI
        )
        _, answer, _, updated_at, _ = contract.functions.latestRoundData().call()
        decimals = contract.functions.decimals().call()
        if time.time() - updated_at > 3600:   # reject stale feeds (> 1 hour old)
            return None
        return Decimal(answer) / Decimal(10 ** decimals)
    except Exception:
        return None


# -- 4. CoinGecko Free Tier ----------------------------------------------------
_CG_BASE = "https://api.coingecko.com/api/v3"
_CG_IDS: Dict[str, str] = {
    "WPOL":   "matic-network",              "POL":    "matic-network",
    "WETH":   "weth",                       "WBTC":   "wrapped-bitcoin",
    "USDC":   "usd-coin",                   "USDC.e": "usd-coin",
    "USDT":   "tether",                     "DAI":    "dai",
    "LINK":   "chainlink",                  "AAVE":   "aave",
    "CRV":    "curve-dao-token",            "UNI":    "uniswap",
    "BAL":    "balancer",                   "FRAX":   "frax",
    "crvUSD": "crvusd",                     "miMATIC":"mimatic",
    "EURS":   "stasis-eurs",                "jEUR":   "jarvis-synthetic-euro",
    "EURT":   "tether-eurt",               "SNX":    "havven",
    "SUSHI":  "sushi",                      "COMP":   "compound-governance-token",
    "YFI":    "yearn-finance",              "GRT":    "the-graph",
    "LDO":    "lido-dao",                   "SAND":   "the-sandbox",
    "MANA":   "decentraland",              "APE":    "apecoin",
    "QUICK":  "quick",                      "RETH":   "rocket-pool-eth",
    "CBETH":  "coinbase-wrapped-staked-eth",
}

def _coingecko_prices(symbols: list) -> Dict[str, Decimal]:
    """Bulk-fetches USD prices via CoinGecko /simple/price (free tier, ~30 req/min)."""
    ids_map = {sym: _CG_IDS[sym] for sym in symbols if sym in _CG_IDS}
    if not ids_map:
        return {}
    headers = {"accept": "application/json"}
    if COINGECKO_KEY:
        headers["x-cg-demo-api-key"] = COINGECKO_KEY
    try:
        resp = requests.get(
            f"{_CG_BASE}/simple/price",
            params={"ids": ",".join(set(ids_map.values())), "vs_currencies": "usd"},
            headers=headers, timeout=8,
        )
        resp.raise_for_status()
        data = resp.json()
        id_to_syms: Dict[str, list] = {}
        for sym, cg_id in ids_map.items():
            id_to_syms.setdefault(cg_id, []).append(sym)
        result: Dict[str, Decimal] = {}
        for cg_id, price_dict in data.items():
            price = Decimal(str(price_dict.get("usd", 0)))
            if price > 0:
                for sym in id_to_syms.get(cg_id, []):
                    result[sym] = price
        return result
    except Exception as e:
        print(f"  \u26a0\ufe0f  CoinGecko unavailable: {e}")
        return {}


# -- 5. DexScreener: Pool TVL & Spot Price ------------------------------------
_DS_BASE = "https://api.dexscreener.com/latest/dex"

def dexscreener_token_price(token_address: str):
    """
    Returns USD spot price for a Polygon token address via DexScreener.
    Picks the highest-liquidity pair for that token. No API key required.
    """
    try:
        resp = requests.get(f"{_DS_BASE}/tokens/{token_address}", timeout=8)
        resp.raise_for_status()
        pairs = [p for p in resp.json().get("pairs", []) if p.get("chainId") == "polygon"]
        if not pairs:
            return None
        best = max(pairs, key=lambda p: float(p.get("liquidity", {}).get("usd", 0) or 0))
        ps = best.get("priceUsd")
        return Decimal(ps) if ps else None
    except Exception:
        return None

def dexscreener_pool_tvl(pair_address: str) -> Decimal:
    """
    Returns live TVL (USD) for a Polygon pool address via DexScreener.
    Used to validate the MIN_POOL_TVL_USD gate before executing flash arb.
    No API key required.
    """
    try:
        resp = requests.get(f"{_DS_BASE}/pairs/polygon/{pair_address}", timeout=8)
        resp.raise_for_status()
        pairs = resp.json().get("pairs", [])
        if not pairs:
            return Decimal("0")
        liq = pairs[0].get("liquidity", {}).get("usd") or 0
        return Decimal(str(liq))
    except Exception:
        return Decimal("0")


# -- 6. 1inch Price API (free tier) --------------------------------------------
_1INCH_PRICE_URL = "https://api.1inch.dev/price/v1.1/137"

def _1inch_prices(symbols: list) -> Dict[str, Decimal]:
    """
    Fetches USD prices from the 1inch Token Price API on Polygon (chain 137).
    Requires a free API key from https://portal.1inch.dev - set ONEINCH_API_KEY.
    Returns empty dict silently when no key is set.
    """
    if not ONEINCH_API_KEY:
        return {}
    addrs = [TOKEN_ADDRESSES[sym] for sym in symbols if sym in TOKEN_ADDRESSES]
    if not addrs:
        return {}
    try:
        resp = requests.post(
            _1INCH_PRICE_URL,
            json={"tokens": addrs, "currency": "USD"},
            headers={"Authorization": "Bearer " + ONEINCH_API_KEY, "accept": "application/json"},
            timeout=8,
        )
        resp.raise_for_status()
        addr_to_price = {k.lower(): Decimal(str(v)) for k, v in resp.json().items()}
        result: Dict[str, Decimal] = {}
        for sym in symbols:
            addr = TOKEN_ADDRESSES.get(sym, "").lower()
            if addr in addr_to_price and addr_to_price[addr] > 0:
                result[sym] = addr_to_price[addr]
        return result
    except Exception as e:
        print(f"  \u26a0\ufe0f  1inch price API unavailable: {e}")
        return {}


# -- 7. Unified Price Refresh --------------------------------------------------
_FALLBACK_PRICES: Dict[str, Decimal] = {
    "USDC":    Decimal("1.00"),   "USDC.e": Decimal("1.00"),
    "USDT":    Decimal("1.00"),   "DAI":    Decimal("1.00"),
    "FRAX":    Decimal("1.00"),   "crvUSD": Decimal("1.00"),  "miMATIC": Decimal("1.00"),
    "WBTC":    Decimal("65000"),  "WETH":   Decimal("3200"),
    "WPOL":    Decimal("0.35"),   "POL":    Decimal("0.35"),
    "LINK":    Decimal("14.00"),  "AAVE":   Decimal("100.00"),
    "CRV":     Decimal("0.45"),   "UNI":    Decimal("8.00"),
    "BAL":     Decimal("3.00"),
    "EURS":    Decimal("1.07"),   "jEUR":   Decimal("1.07"),
    "PAR":     Decimal("1.07"),   "EURT":   Decimal("1.07"),
}

TOKEN_USD_PRICE: Dict[str, Decimal] = dict(_FALLBACK_PRICES)
_PRICE_LAST_REFRESH = 0.0
PRICE_TTL_SECONDS   = 60       # minimum seconds between live refreshes

def refresh_token_prices(force: bool = False) -> Dict[str, Decimal]:
    """
    Refreshes TOKEN_USD_PRICE from live sources in priority order:
      1. CoinGecko free tier  - broad coverage, no key required
      2. 1inch Price API      - DeFi-native spot prices (requires free API key)
      3. Chainlink on-chain   - highest authority, overrides REST for covered symbols
    Hardcoded fallbacks cover any symbol not reached by live sources.
    Results are cached for PRICE_TTL_SECONDS to respect rate limits.
    """
    global TOKEN_USD_PRICE, _PRICE_LAST_REFRESH
    if not force and (time.time() - _PRICE_LAST_REFRESH) < PRICE_TTL_SECONDS:
        return TOKEN_USD_PRICE

    updated: Dict[str, Decimal] = dict(_FALLBACK_PRICES)

    # Layer 1 -- CoinGecko (no key required, ~30 symbols in one call)
    cg = _coingecko_prices(list(_CG_IDS.keys()))
    updated.update(cg)
    if cg:
        print(f"  \U0001f4e1 CoinGecko:  {len(cg)} prices fetched")

    # Layer 2 -- 1inch (DeFi-native; requires free API key from portal.1inch.dev)
    inch = _1inch_prices(list(TOKEN_ADDRESSES.keys()))
    updated.update(inch)
    if inch:
        print(f"  \U0001f504 1inch:       {len(inch)} prices fetched")

    # Layer 3 -- Chainlink on-chain (most authoritative; overrides REST sources)
    cl_count = 0
    for sym in list(CHAINLINK_FEEDS.keys()):
        price = _chainlink_price(sym)
        if price and price > 0:
            updated[sym] = price
            cl_count += 1
    if cl_count:
        print(f"  \U0001f517 Chainlink:  {cl_count} on-chain prices confirmed")

    TOKEN_USD_PRICE = updated
    _PRICE_LAST_REFRESH = time.time()
    return TOKEN_USD_PRICE


# -- 8. Reload LIVE_POOLS with on-chain data if this cell upgraded RPC ----------
# Cell 6 loads LIVE_POOLS before WSS is established; re-fetch when RPC is now live.
_pools_are_simulated = all("address" not in p for p in LIVE_POOLS.values())
if RPC_LIVE and _pools_are_simulated:
    print("\n🔄 RPC now live — reloading pool states with on-chain data...")
    LIVE_POOLS = load_all_live_pools(DEEP_POOL_REGISTRY)
    print(f"📦 LIVE_POOLS reloaded: {len(LIVE_POOLS)} pools across "
          f"{len({p['protocol'] for p in LIVE_POOLS.values()})} protocol families")

# Initial price refresh on cell execution
print("\n\U0001f30d Refreshing token prices from live oracles...")
refresh_token_prices(force=True)
print(f"\n\u2705 TOKEN_USD_PRICE ready  |  {len(TOKEN_USD_PRICE)} tokens")
_preview = [(s, float(TOKEN_USD_PRICE[s])) for s in ["WETH", "WBTC", "WPOL", "LINK", "AAVE"] if s in TOKEN_USD_PRICE]
for _sym, _px in _preview:
    print(f"   {_sym:<8} ${_px:,.2f}")


In [ ]:
# ==============================================================================
# CELL 7: FLASH LOAN CAPITAL LAYER
# Models Aave V3 and Balancer Vault flash liquidity sources on Polygon.
# Computes available capital, flash fees, and net repayment obligations.
# ==============================================================================
from dataclasses import dataclass
from enum import Enum

# ── Flash Loan Source Configuration ──────────────────────────────────────────
AAVE_V3_POOL_POLYGON    = "0x794a61358D6845594F94dc1DB02A252b5b4814aD"
BALANCER_VAULT_POLYGON  = "0xBA12222222228d8Ba445958a75a0704d566BF2C8"

# Aave V3 flash loan fee: 0.05 % of principal
AAVE_FLASH_FEE_BPS      = Decimal("5")      # 0.05 %
# Balancer flash loan fee: 0 % (Balancer charges zero flash fee on Polygon)
BALANCER_FLASH_FEE_BPS  = Decimal("0")

# ── Gas & USD Parameters ─────────────────────────────────────────────────────
GAS_PRICE_GWEI          = Decimal("50")        # Conservative Polygon gas price
GAS_UNITS_SIMPLE_ARB    = Decimal("350000")    # Two-hop flash arb
GAS_UNITS_THREE_HOP_ARB = Decimal("500000")    # Three-hop flash arb
POL_USD_PRICE           = Decimal("0.35")      # Approximate; replace with oracle

GAS_COST_USD_2HOP  = (GAS_PRICE_GWEI * Decimal("1e-9") * GAS_UNITS_SIMPLE_ARB    * POL_USD_PRICE)
GAS_COST_USD_3HOP  = (GAS_PRICE_GWEI * Decimal("1e-9") * GAS_UNITS_THREE_HOP_ARB * POL_USD_PRICE)

# ── Profitability Gate Constants ─────────────────────────────────────────────
MIN_POOL_TVL_USD        = Decimal("50000")    # $50k minimum pool TVL
MIN_NET_PROFIT_USD      = Decimal("5.00")     # $5.00 minimum after all costs
MIN_PROFIT_TO_GAS_RATIO = Decimal("1.10")     # 10 % profit cushion above gas
RELAY_TIP_USD           = Decimal("0.50")     # Private mempool tip estimate
RISK_BUFFER_USD         = Decimal("1.00")     # Slippage / revert risk reserve


class FlashSource(str, Enum):
    AAVE      = "Aave_V3"
    BALANCER  = "Balancer_Vault"


@dataclass
class FlashLoanParams:
    source:         FlashSource
    asset:          str              # token symbol
    principal_usd:  Decimal
    fee_usd:        Decimal
    repayment_usd:  Decimal          # principal + fee


@dataclass
class Profitability:
    gross_amount_out:   Decimal      # raw output from math engine
    flashloan:          FlashLoanParams
    gas_cost_usd:       Decimal
    relay_tip_usd:      Decimal
    risk_buffer_usd:    Decimal
    net_profit_usd:     Decimal
    profit_to_gas:      Decimal
    passes_gate:        bool


def compute_flash_params(
    principal_usd: Decimal,
    source: FlashSource = FlashSource.BALANCER,
    asset: str = "USDC",
) -> FlashLoanParams:
    """Returns flash loan cost structure for a given principal and source."""
    fee_bps = BALANCER_FLASH_FEE_BPS if source == FlashSource.BALANCER else AAVE_FLASH_FEE_BPS
    fee_usd = principal_usd * fee_bps / Decimal("10000")
    return FlashLoanParams(
        source=source,
        asset=asset,
        principal_usd=principal_usd,
        fee_usd=fee_usd,
        repayment_usd=principal_usd + fee_usd,
    )


def evaluate_profitability(
    gross_amount_out_usd: Decimal,
    principal_usd: Decimal,
    hops: int = 2,
    flash_source: FlashSource = FlashSource.BALANCER,
    asset: str = "USDC",
) -> Profitability:
    """
    Applies the full net-profit formula:
        net_profit = gross_out - principal - flash_fee - gas - relay_tip - risk_buffer
    """
    flash = compute_flash_params(principal_usd, flash_source, asset)
    gas_usd = GAS_COST_USD_3HOP if hops >= 3 else GAS_COST_USD_2HOP

    net = (gross_amount_out_usd
           - flash.repayment_usd
           - gas_usd
           - RELAY_TIP_USD
           - RISK_BUFFER_USD)

    profit_to_gas = net / gas_usd if gas_usd > 0 else Decimal("0")
    passes = (net >= MIN_NET_PROFIT_USD) and (profit_to_gas >= MIN_PROFIT_TO_GAS_RATIO)

    return Profitability(
        gross_amount_out=gross_amount_out_usd,
        flashloan=flash,
        gas_cost_usd=gas_usd,
        relay_tip_usd=RELAY_TIP_USD,
        risk_buffer_usd=RISK_BUFFER_USD,
        net_profit_usd=net,
        profit_to_gas=profit_to_gas,
        passes_gate=passes,
    )


# ── Self-test with a representative opportunity ───────────────────────────────
_test = evaluate_profitability(
    gross_amount_out_usd=Decimal("10250"),   # buy 10k, get 10250 out
    principal_usd=Decimal("10000"),
    hops=2,
    flash_source=FlashSource.BALANCER,
)
print("⚡ Flash Loan Capital Layer initialised")
print(f"   Flash source  : {_test.flashloan.source.value}")
print(f"   Principal     : ${_test.flashloan.principal_usd:,.2f}")
print(f"   Flash fee     : ${_test.flashloan.fee_usd:,.4f}")
print(f"   Gas cost      : ${_test.gas_cost_usd:.4f}")
print(f"   Net profit    : ${_test.net_profit_usd:.4f}  ({'✅ PASS' if _test.passes_gate else '❌ FAIL'})")
print(f"   Profit/Gas    : {float(_test.profit_to_gas):.3f}x")


In [ ]:
# ==============================================================================
# CELL 8: LIVE ROUTE SCANNER — NET-PROFIT GATED OPPORTUNITY RANKER
# Feeds LIVE_POOLS into the pricing and arbitrage engines, then applies the
# full profitability gate.  Every opportunity that survives is ranked by
# net_profit_usd descending and stored in RANKED_OPPORTUNITIES.
# ==============================================================================
from dataclasses import dataclass, field
from typing import Optional

# ── Live USD oracle (populated by Cell 6B via refresh_token_prices()) ──────────
refresh_token_prices()    # uses cached values if TTL not expired
DEFAULT_TOKEN_USD = Decimal("1.00")

def token_price_usd(symbol: str) -> Decimal:
    return TOKEN_USD_PRICE.get(symbol, DEFAULT_TOKEN_USD)


@dataclass
class LiveOpportunity:
    """A ranked, gate-passed arbitrage opportunity ready for simulation."""
    opp_id:         str
    path:           List[str]
    pool_sequence:  List[str]
    protocol_seq:   List[str]
    gross_rate:     Decimal
    gross_out_usd:  Decimal
    profitability:  Profitability
    block_detected: int
    flash_source:   FlashSource


def _best_pool_for_pair(t_in: str, t_out: str, rates: dict, pools: dict) -> Optional[str]:
    """Returns the pool_id with the highest rate for (t_in, t_out), or None."""
    entries = rates.get((t_in, t_out), [])
    for entry in entries:          # already sorted best-first
        if entry["pool_id"] in pools:
            return entry["pool_id"]
    return None


def score_opportunities(
    arb_cycles: List[dict],
    pools: dict,
    rates: dict,
    principal_usd: Decimal = Decimal("10000"),
    flash_source: FlashSource = FlashSource.BALANCER,
) -> List["LiveOpportunity"]:
    """
    Takes raw Bellman-Ford cycles, re-derives the best pool per hop from
    the live rate matrix, computes hop-by-hop token amounts, applies the
    profitability gate, and returns ranked LiveOpportunity list.
    """
    live_ops: List[LiveOpportunity] = []
    opp_counter = 0

    for cycle in arb_cycles:
        path = cycle["path"]          # [t0, t1, ..., t0]
        hops = len(path) - 1

        borrow_asset = path[0]
        borrow_price = token_price_usd(borrow_asset)
        principal_tokens = principal_usd / borrow_price if borrow_price > 0 else Decimal("10000")

        amount     = principal_tokens
        pool_seq   = []
        proto_seq  = []
        valid      = True

        for hop_idx in range(hops):
            t_in  = path[hop_idx]
            t_out = path[hop_idx + 1]

            pid = _best_pool_for_pair(t_in, t_out, rates, pools)
            if pid is None:
                valid = False
                break

            pool = pools[pid]
            tok_list = pool.get("tokens", [])
            i = tok_list.index(t_in)  if t_in  in tok_list else 0
            j = tok_list.index(t_out) if t_out in tok_list else (1 if i == 0 else 0)

            proto = pool["protocol"]
            pool_seq.append(pid)
            proto_seq.append(proto)

            if proto == "UniswapV2":
                amount = DeFiEngineMath.query_uniswap_v2(
                    pool["reserves"][i], pool["reserves"][j], amount, pool["fee"])
            elif proto == "UniswapV3":
                amount = DeFiEngineMath.query_uniswap_v3(
                    pool["sqrtPriceX96"], pool["liquidity"], amount,
                    i == 0, pool["fee_bps"])
            elif proto == "Curve":
                ains = [Decimal("0")] * len(pool["reserves"])
                ains[i] = amount
                amount = DeFiEngineMath.query_curve_stable(pool["reserves"], ains, i, j, pool["A"])
            elif proto == "Balancer":
                amount = DeFiEngineMath.query_balancer_weighted(
                    pool["reserves"], pool["weights"], amount, i, j, pool["swap_fee"])

            if amount <= 0:
                valid = False
                break

        if not valid:
            continue

        gross_out_usd = amount * borrow_price
        prof = evaluate_profitability(gross_out_usd, principal_usd, hops, flash_source, borrow_asset)

        if not prof.passes_gate:
            continue

        opp_counter += 1
        live_ops.append(LiveOpportunity(
            opp_id        = f"OPP-{opp_counter:04d}",
            path          = path,
            pool_sequence = pool_seq,
            protocol_seq  = proto_seq,
            gross_rate    = cycle["cumulative_rate"],
            gross_out_usd = gross_out_usd,
            profitability = prof,
            block_detected= BLOCK,
            flash_source  = flash_source,
        ))

    live_ops.sort(key=lambda x: x.profitability.net_profit_usd, reverse=True)
    return live_ops


def print_live_opportunities(ops: List[LiveOpportunity]):
    if not ops:
        print("  ↳ No opportunities survive the profitability gate at current pool state.")
        return

    print(f"\n{'='*90}")
    print(f"💰 LIVE RANKED OPPORTUNITIES — {len(ops)} passed profitability gate")
    print(f"{'='*90}")
    for op in ops:
        path_str = " → ".join(op.path)
        p = op.profitability
        fl = p.flashloan
        tier = "🟢 STRONG" if float(p.net_profit_usd) > 20 else ("🟡 MARGINAL" if float(p.net_profit_usd) > 5 else "🔴 MICRO")
        print(f"\n  {op.opp_id}  {tier}")
        print(f"  Path       : {path_str}")
        print(f"  Pools      : {' → '.join(op.pool_sequence)}")
        print(f"  Flash Src  : {fl.source.value}  ({op.gross_rate:.6f}x gross rate)")
        print(f"  Principal  : ${fl.principal_usd:,.2f}  |  Flash Fee : ${fl.fee_usd:.4f}")
        print(f"  Gross Out  : ${p.gross_amount_out:,.4f}")
        print(f"  Gas        : ${p.gas_cost_usd:.4f}  |  Relay: ${p.relay_tip_usd}  |  Risk Buffer: ${p.risk_buffer_usd}")
        print(f"  NET PROFIT : ${p.net_profit_usd:.4f}  |  Profit/Gas: {float(p.profit_to_gas):.3f}x")
        print(f"  Block      : #{op.block_detected:,}")
    print(f"\n{'='*90}")


# ── Run the full pipeline on live (or simulation) pools ───────────────────────
print("🔁 Computing cross-pool rates from LIVE_POOLS...")
LIVE_RATES = compute_all_pool_rates(LIVE_POOLS)

print("⚡ Running Bellman-Ford on live rate graph...")
_live_arb_engine = ArbitrageGraphEngine(LIVE_RATES)
_raw_cycles      = _live_arb_engine.bellman_ford_all_sources()
print(f"   ↳ {len(_raw_cycles)} raw cycles detected before gate")

RANKED_OPPORTUNITIES = score_opportunities(_raw_cycles, LIVE_POOLS, LIVE_RATES)
print(f"   ↳ {len(RANKED_OPPORTUNITIES)} opportunities pass the profitability gate")
print_live_opportunities(RANKED_OPPORTUNITIES)


In [ ]:
# ==============================================================================
# CELL 9: C1/C2 FULL STATE MACHINE
#
# C1 executes against the pre-trade pool state.
# After C1 confirms on-chain, pools mutate.
# C2 reloads the confirmed post-C1 state, independently recomputes
# profitability, and decides: MIRROR | REVERSE | DO_NOTHING.
# C2 is bounded to ~5 blocks (~10 seconds on Polygon).
#
# Lifecycle:
#   Opportunity detected
#       ↓
#   C1Cycle created (PRE_CHECK → SIMULATING → SUBMITTED → CONFIRMED | FAILED)
#       ↓
#   C1 tx confirms → pools mutate
#       ↓
#   C2Cycle created from post-C1 snapshot
#   C2 recomputes route within 5-block window
#       ↓
#   C2Decision: MIRROR | REVERSE | DO_NOTHING
#       ↓
#   C2 submitted if MIRROR or REVERSE
# ==============================================================================
from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Optional
import time

BLOCKS_C2_WINDOW = 5        # Maximum blocks C2 may act after C1 confirmation
BLOCK_TIME_S     = 2.0      # Polygon average block time in seconds


class OppStatus(Enum):
    DETECTED   = auto()
    EVALUATING = auto()
    GATE_PASS  = auto()
    GATE_FAIL  = auto()
    EXPIRED    = auto()


class C1Status(Enum):
    PRE_CHECK  = auto()
    SIMULATING = auto()
    SIM_FAIL   = auto()
    SUBMITTED  = auto()
    CONFIRMED  = auto()
    FAILED     = auto()
    REVERTED   = auto()


class C2Decision(Enum):
    PENDING    = auto()
    MIRROR     = auto()     # Same direction — spread persists post-C1
    REVERSE    = auto()     # Opposite direction — C1 moved price, new spread emerged
    DO_NOTHING = auto()     # No profitable edge found within C2 window


class C2Status(Enum):
    PENDING    = auto()
    COMPUTING  = auto()
    SUBMITTED  = auto()
    CONFIRMED  = auto()
    FAILED     = auto()
    EXPIRED    = auto()


@dataclass
class PoolSnapshot:
    """Immutable point-in-time pool state captured at a specific block."""
    pool_id:   str
    protocol:  str
    tokens:    List[str]
    reserves:  List[Decimal]
    block:     int
    timestamp: float = field(default_factory=time.time)

    # V3-specific
    sqrt_price_x96: Optional[Decimal] = None
    liquidity:      Optional[Decimal] = None
    fee_bps:        Optional[int]     = None

    # Balancer-specific
    weights: Optional[List[Decimal]] = None
    swap_fee: Optional[Decimal]      = None

    # Curve-specific
    A: Optional[Decimal]             = None


def snapshot_pool(pool_id: str, pool: dict, block: int) -> PoolSnapshot:
    """Captures a PoolSnapshot from a live pool dict at the given block."""
    return PoolSnapshot(
        pool_id  = pool_id,
        protocol = pool["protocol"],
        tokens   = list(pool["tokens"]),
        reserves = list(pool.get("reserves", [])),
        block    = block,
        sqrt_price_x96 = pool.get("sqrtPriceX96"),
        liquidity      = pool.get("liquidity"),
        fee_bps        = pool.get("fee_bps"),
        weights        = pool.get("weights"),
        swap_fee       = pool.get("swap_fee"),
        A              = pool.get("A"),
    )


def snapshot_to_pool_dict(snap: PoolSnapshot) -> dict:
    """Reconstructs a pool dict from a snapshot for use with DeFiEngineMath."""
    d = {"protocol": snap.protocol, "tokens": snap.tokens,
         "reserves": snap.reserves}
    if snap.protocol == "UniswapV2":
        d["fee"] = snap.swap_fee or Decimal("0.003")
    elif snap.protocol == "UniswapV3":
        d["sqrtPriceX96"] = snap.sqrt_price_x96
        d["liquidity"]    = snap.liquidity
        d["fee_bps"]      = snap.fee_bps
    elif snap.protocol == "Curve":
        d["A"] = snap.A or Decimal("100")
    elif snap.protocol == "Balancer":
        d["weights"]  = snap.weights
        d["swap_fee"] = snap.swap_fee or Decimal("0.0025")
    return d


@dataclass
class C1Cycle:
    c1_id:      str
    opportunity: "LiveOpportunity"
    status:     C1Status = C1Status.PRE_CHECK

    # Captured pre-execution pool snapshots for each pool in the route
    pre_snapshots: Dict[str, PoolSnapshot] = field(default_factory=dict)

    # On-chain results
    tx_hash:       Optional[str]     = None
    block_confirmed: Optional[int]   = None
    gas_used:      Optional[int]     = None
    actual_profit_usd: Optional[Decimal] = None

    created_at:    float = field(default_factory=time.time)

    def capture_pre_state(self, pools: dict, block: int):
        for pid in self.opportunity.pool_sequence:
            if pid in pools:
                self.pre_snapshots[pid] = snapshot_pool(pid, pools[pid], block)

    def mark_confirmed(self, tx_hash: str, block: int, gas_used: int, profit_usd: Decimal):
        self.tx_hash           = tx_hash
        self.block_confirmed   = block
        self.gas_used          = gas_used
        self.actual_profit_usd = profit_usd
        self.status            = C1Status.CONFIRMED


@dataclass
class C2Cycle:
    c2_id:      str
    c1_cycle:   C1Cycle
    decision:   C2Decision = C2Decision.PENDING
    status:     C2Status   = C2Status.PENDING

    # Post-C1 pool snapshots loaded after C1 confirms
    post_snapshots: Dict[str, PoolSnapshot] = field(default_factory=dict)

    # Recomputed opportunity after pool mutation
    recomputed_opportunity: Optional["LiveOpportunity"] = None

    tx_hash:       Optional[str]     = None
    block_confirmed: Optional[int]   = None
    actual_profit_usd: Optional[Decimal] = None

    created_at:    float = field(default_factory=time.time)
    deadline_block: int  = 0

    def set_deadline(self):
        self.deadline_block = (self.c1_cycle.block_confirmed or BLOCK) + BLOCKS_C2_WINDOW

    def is_within_window(self, current_block: int) -> bool:
        return current_block <= self.deadline_block

    def capture_post_state(self, pools: dict, block: int):
        """Load fresh pool states after C1 has mutated on-chain reserves."""
        for pid in self.c1_cycle.opportunity.pool_sequence:
            if RPC_LIVE:
                pool_meta = DEEP_POOL_REGISTRY.get(pid)
                if pool_meta:
                    fresh = load_live_pool_state(pid, pool_meta)
                    if fresh:
                        pools[pid] = fresh
            if pid in pools:
                self.post_snapshots[pid] = snapshot_pool(pid, pools[pid], block)

    def recompute(self, pools: dict, current_block: int) -> C2Decision:
        """
        Independently recomputes profitability using post-C1 pool state.
        Evaluates original direction (MIRROR) and reversed path (REVERSE).
        Returns the best decision within the C2 window.
        """
        self.status = C2Status.COMPUTING

        if not self.is_within_window(current_block):
            self.decision = C2Decision.DO_NOTHING
            self.status   = C2Status.EXPIRED
            return self.decision

        post_pool_dict = {pid: snapshot_to_pool_dict(snap)
                         for pid, snap in self.post_snapshots.items()}

        orig_opp = self.c1_cycle.opportunity

        # ── Test MIRROR: same path, same direction ────────────────────────
        mirror_rate  = _walk_path(orig_opp.path, orig_opp.pool_sequence, post_pool_dict)
        mirror_gross = mirror_rate * (orig_opp.profitability.flashloan.principal_usd
                                       / token_price_usd(orig_opp.path[0]))
        mirror_gross_usd = mirror_gross * token_price_usd(orig_opp.path[0])
        mirror_prof  = evaluate_profitability(
            mirror_gross_usd, orig_opp.profitability.flashloan.principal_usd,
            len(orig_opp.path) - 1, orig_opp.flash_source, orig_opp.path[0])

        # ── Test REVERSE: flip the path ───────────────────────────────────
        rev_path     = list(reversed(orig_opp.path))
        rev_pool_seq = list(reversed(orig_opp.pool_sequence))
        reverse_rate = _walk_path(rev_path, rev_pool_seq, post_pool_dict)
        reverse_gross_usd = reverse_rate * token_price_usd(rev_path[0])
        reverse_prof = evaluate_profitability(
            reverse_gross_usd, orig_opp.profitability.flashloan.principal_usd,
            len(rev_path) - 1, orig_opp.flash_source, rev_path[0])

        best_decision  = C2Decision.DO_NOTHING
        best_net       = Decimal("0")

        if mirror_prof.passes_gate and mirror_prof.net_profit_usd > best_net:
            best_decision = C2Decision.MIRROR
            best_net      = mirror_prof.net_profit_usd

        if reverse_prof.passes_gate and reverse_prof.net_profit_usd > best_net:
            best_decision = C2Decision.REVERSE
            best_net      = reverse_prof.net_profit_usd

        self.decision = best_decision
        if best_decision != C2Decision.DO_NOTHING:
            self.status = C2Status.PENDING   # ready for submission
        return self.decision


def _walk_path(path: List[str], pool_seq: List[str], pools: dict) -> Decimal:
    """Simulates the amount-out walk along a token path using given pool states."""
    amount = Decimal("10000")
    for hop_idx, pid in enumerate(pool_seq):
        pool = pools.get(pid)
        if not pool:
            return Decimal("0")
        t_in  = path[hop_idx]
        t_out = path[hop_idx + 1] if hop_idx + 1 < len(path) else path[0]
        tok_list = pool.get("tokens", [])
        i = tok_list.index(t_in)  if t_in  in tok_list else 0
        j = tok_list.index(t_out) if t_out in tok_list else 1
        proto = pool["protocol"]
        if proto == "UniswapV2":
            amount = DeFiEngineMath.query_uniswap_v2(pool["reserves"][i], pool["reserves"][j], amount, pool["fee"])
        elif proto == "UniswapV3":
            amount = DeFiEngineMath.query_uniswap_v3(pool["sqrtPriceX96"], pool["liquidity"], amount, i==0, pool["fee_bps"])
        elif proto == "Curve":
            ains = [Decimal("0")] * len(pool["reserves"])
            ains[i] = amount
            amount = DeFiEngineMath.query_curve_stable(pool["reserves"], ains, i, j, pool["A"])
        elif proto == "Balancer":
            amount = DeFiEngineMath.query_balancer_weighted(pool["reserves"], pool["weights"], amount, i, j, pool["swap_fee"])
        if amount <= 0:
            return Decimal("0")
    return amount


# ── State Machine Demo ────────────────────────────────────────────────────────
print("🤖 C1/C2 State Machine — System-wide definition complete")
print(f"   C2 window     : {BLOCKS_C2_WINDOW} blocks (~{int(BLOCKS_C2_WINDOW * BLOCK_TIME_S)}s on Polygon)")
print(f"   C2 decisions  : {[d.name for d in C2Decision]}")
print(f"   C1 statuses   : {[s.name for s in C1Status]}")
print(f"   C2 statuses   : {[s.name for s in C2Status]}")

if RANKED_OPPORTUNITIES:
    top_opp = RANKED_OPPORTUNITIES[0]
    demo_c1 = C1Cycle(c1_id="C1-0001", opportunity=top_opp)
    demo_c1.capture_pre_state(LIVE_POOLS, BLOCK)
    print(f"\n  Demo C1 cycle created for {top_opp.opp_id}")
    print(f"  Path          : {' → '.join(top_opp.path)}")
    print(f"  Status        : {demo_c1.status.name}")
    print(f"  Pre-snapshots : {list(demo_c1.pre_snapshots.keys())}")

    # Simulate C1 confirmation (in production: wait for tx receipt)
    demo_c1.mark_confirmed(
        tx_hash="0x" + "ab" * 32,
        block=BLOCK + 1,
        gas_used=320000,
        profit_usd=top_opp.profitability.net_profit_usd,
    )
    print(f"  C1 Confirmed  : block #{demo_c1.block_confirmed}")

    # Spawn C2
    demo_c2 = C2Cycle(c2_id="C2-0001", c1_cycle=demo_c1)
    demo_c2.set_deadline()
    demo_c2.capture_post_state(LIVE_POOLS, demo_c1.block_confirmed)
    decision = demo_c2.recompute(LIVE_POOLS, demo_c1.block_confirmed + 1)
    print(f"\n  C2 Decision   : {decision.name}")
    print(f"  C2 Deadline   : block #{demo_c2.deadline_block}")
    print(f"  C2 Status     : {demo_c2.status.name}")
else:
    print("\n  ↳ No ranked opportunities available for C1/C2 demo (simulation mode).")
    print("     Provide POLYGON_RPC_URL with live data to generate live opportunities.")

print("\n✅ C1/C2 state machine validated system-wide.")


In [ ]:
# ==============================================================================
# CELL 10: TRANSACTION BUILDER & GUARDED EXECUTION
#
# Constructs EIP-1559 flash-arb calldata, verifies wallet balances and
# approvals, and submits to Polygon mainnet ONLY when all three execution
# guards are set:
#
#   EXECUTION_MODE=live
#   LIVE_TRADING=1
#   CONFIRM_MAINNET_EXECUTION=I_UNDERSTAND_POLYGON_MAINNET_RISK
#
# Without all three guards the system prints the transaction plan and halts.
# Private key and executor address are loaded from environment variables only.
# ==============================================================================

# ── Execution Guard Evaluation ────────────────────────────────────────────────
EXEC_MODE    = os.environ.get("EXECUTION_MODE",              "simulation")
LIVE_FLAG    = os.environ.get("LIVE_TRADING",                "0")
CONFIRM_FLAG = os.environ.get("CONFIRM_MAINNET_EXECUTION",   "")
REQUIRED_CONFIRM = "I_UNDERSTAND_POLYGON_MAINNET_RISK"

EXECUTION_ARMED = (
    EXEC_MODE    == "live"            and
    LIVE_FLAG    == "1"               and
    CONFIRM_FLAG == REQUIRED_CONFIRM  and
    RPC_LIVE
)

# ── Wallet & Executor Configuration (from environment only) ──────────────────
PRIVATE_KEY       = os.environ.get("EXECUTOR_PRIVATE_KEY",  "")
EXECUTOR_CONTRACT = os.environ.get("EXECUTOR_CONTRACT_ADDR","")
WALLET_ADDR       = ""
if PRIVATE_KEY and w3:
    try:
        from eth_account import Account
        acct = Account.from_key(PRIVATE_KEY)
        WALLET_ADDR = acct.address
    except Exception:
        pass

# ── Minimal Executor ABI (flash-arb entry point) ─────────────────────────────
ABI_EXECUTOR = [
    {
        "name": "executeFlashArb",
        "type": "function",
        "stateMutability": "nonpayable",
        "inputs": [
            {"name": "flashSource",    "type": "uint8"},   # 0=Aave, 1=Balancer
            {"name": "asset",          "type": "address"},
            {"name": "amount",         "type": "uint256"},
            {"name": "poolSequence",   "type": "address[]"},
            {"name": "tokenPath",      "type": "address[]"},
            {"name": "minProfit",      "type": "uint256"},
        ],
        "outputs": [{"name": "profit", "type": "uint256"}],
    }
]


def build_tx_payload(op: "LiveOpportunity", nonce: int, base_fee_gwei: Decimal) -> dict:
    """
    Constructs an EIP-1559 transaction dict for the given opportunity.
    Does NOT sign or send — purely data construction.
    """
    asset_sym   = op.path[0]
    asset_addr  = TOKEN_ADDRESSES.get(asset_sym, "0x" + "00" * 20)
    flash_enum  = 0 if op.flash_source == FlashSource.AAVE else 1
    principal_wei = int(op.profitability.flashloan.principal_usd * Decimal("1e6"))  # USDC 6 decimals
    min_profit_wei = int(MIN_NET_PROFIT_USD * Decimal("1e6"))

    pool_addrs  = [DEEP_POOL_REGISTRY.get(pid, {}).get("address", "0x" + "00" * 20)
                   for pid in op.pool_sequence]
    token_addrs = [TOKEN_ADDRESSES.get(t, "0x" + "00" * 20) for t in op.path]

    max_fee       = int((base_fee_gwei + Decimal("30")) * Decimal("1e9"))  # +30 gwei tip
    priority_fee  = int(Decimal("30") * Decimal("1e9"))

    executor_addr = Web3.to_checksum_address(EXECUTOR_CONTRACT) if EXECUTOR_CONTRACT else "0x" + "00" * 20

    if w3 and EXECUTOR_CONTRACT:
        contract  = w3.eth.contract(address=executor_addr, abi=ABI_EXECUTOR)
        call_data = contract.encodeABI(
            fn_name="executeFlashArb",
            args=[flash_enum, Web3.to_checksum_address(asset_addr),
                  principal_wei,
                  [Web3.to_checksum_address(a) for a in pool_addrs],
                  [Web3.to_checksum_address(a) for a in token_addrs],
                  min_profit_wei],
        )
    else:
        call_data = "0x"  # placeholder when contract address not configured

    return {
        "chainId":              CHAIN_ID,
        "nonce":                nonce,
        "to":                   executor_addr,
        "value":                0,
        "data":                 call_data,
        "maxFeePerGas":         max_fee,
        "maxPriorityFeePerGas": priority_fee,
        "gas":                  500_000,
        "type":                 2,          # EIP-1559
    }


def submit_opportunity(op: "LiveOpportunity", c1: C1Cycle) -> Optional[str]:
    """
    Full guarded submission pipeline:
    1. Build tx payload
    2. Verify wallet balance
    3. Sign transaction
    4. Send to Polygon RPC
    5. Wait for receipt and verify success
    Returns tx_hash on success, None otherwise.
    """
    if not EXECUTION_ARMED:
        return None

    try:
        nonce    = w3.eth.get_transaction_count(WALLET_ADDR)
        base_fee = Decimal(str(w3.eth.gas_price)) / Decimal("1e9")
        tx       = build_tx_payload(op, nonce, base_fee)

        from eth_account import Account
        acct    = Account.from_key(PRIVATE_KEY)
        signed  = acct.sign_transaction(tx)
        tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction).hex()
        receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=60)

        if receipt.status == 1:
            gas_used    = receipt.gasUsed
            gas_cost    = Decimal(gas_used) * base_fee * Decimal("1e-9") * POL_USD_PRICE
            actual_prof = op.gross_out_usd - op.profitability.flashloan.repayment_usd - gas_cost
            c1.mark_confirmed(tx_hash, receipt.blockNumber, gas_used, actual_prof)
            return tx_hash
        else:
            c1.status = C1Status.REVERTED
            return None

    except Exception as exc:
        c1.status = C1Status.FAILED
        print(f"  ❌ Submission failed: {exc}")
        return None


# ── Master Execution Loop ─────────────────────────────────────────────────────
def run_execution_loop(opportunities: List["LiveOpportunity"]):
    """
    Iterates ranked opportunities in priority order.
    In simulation mode: prints full tx plans.
    In live mode: submits, confirms, spawns C2.
    """
    print(f"\n{'='*90}")
    print(f"🚀 EXECUTION LOOP  |  Mode: {'⚠️  LIVE MAINNET' if EXECUTION_ARMED else '🔬 SIMULATION'}  |  Wallet: {WALLET_ADDR or 'NOT SET'}")
    print(f"{'='*90}")

    if not opportunities:
        print("  ↳ No opportunities in queue.")
        return

    for idx, op in enumerate(opportunities[:5]):   # Cap at top-5 per cycle
        print(f"\n  [{idx+1}] {op.opp_id}  |  Net: ${op.profitability.net_profit_usd:.4f}  |  Path: {' → '.join(op.path)}")

        # Build transaction plan
        tx_plan = build_tx_payload(op, nonce=idx, base_fee_gwei=Decimal("50"))
        print(f"       Flash source : {op.flash_source.value}")
        print(f"       Pool sequence: {op.pool_sequence}")
        print(f"       Gas limit    : {tx_plan['gas']:,}")
        print(f"       Max fee      : {tx_plan['maxFeePerGas'] / 1e9:.1f} gwei")
        print(f"       Calldata     : {str(tx_plan['data'])[:66]}{'...' if len(str(tx_plan['data'])) > 66 else ''}")

        if EXECUTION_ARMED:
            c1 = C1Cycle(c1_id=f"C1-{idx+1:04d}", opportunity=op)
            c1.capture_pre_state(LIVE_POOLS, BLOCK)
            c1.status = C1Status.SIMULATING

            tx_hash = submit_opportunity(op, c1)
            if tx_hash:
                print(f"       ✅ C1 CONFIRMED  tx={tx_hash[:20]}...  block={c1.block_confirmed}")
                c2 = C2Cycle(c2_id=f"C2-{idx+1:04d}", c1_cycle=c1)
                c2.set_deadline()
                c2.capture_post_state(LIVE_POOLS, c1.block_confirmed)
                decision = c2.recompute(LIVE_POOLS, c1.block_confirmed + 1)
                print(f"       C2 Decision : {decision.name}  (deadline block #{c2.deadline_block})")
                if decision in (C2Decision.MIRROR, C2Decision.REVERSE):
                    print(f"       🔄 C2 submitted ({decision.name})")
            else:
                print(f"       ❌ C1 failed or reverted. Status: {c1.status.name}")
        else:
            guards = {
                "EXECUTION_MODE=live":                  EXEC_MODE    == "live",
                "LIVE_TRADING=1":                       LIVE_FLAG    == "1",
                f"CONFIRM_MAINNET_EXECUTION={REQUIRED_CONFIRM}": CONFIRM_FLAG == REQUIRED_CONFIRM,
                "RPC_LIVE":                             RPC_LIVE,
            }
            missing = [k for k, v in guards.items() if not v]
            print(f"       🔒 SIMULATION ONLY — missing guards: {missing}")

    print(f"\n{'='*90}")
    print("✅ Execution loop complete.")
    print(f"{'='*90}\n")


run_execution_loop(RANKED_OPPORTUNITIES)
